# Count the number of colocalization points in the Mediterranean sea
## What is done here ?
- select the drifters points under the swaths for each cycle considering drifters points for which it is the nearest swot image (one cycle every 23h50 approximately, e.g drifters in the $\pm$ 12h25 before and after the swot image)
- compute the minimum/maximum time gap for each swot image/cycle (will depend on the dt of the regular grid)

## Results ?
- some cycle are not available :
    - swath 3 : 498, 526-528, 566, 572
    - swath 16 : 480, 508, 513, 526-528, 534

In [1]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir
from swot import browse_swot_250, add_mask_inside_swot, build_swath_polygon

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pyproj
from rasterio.transform import Affine

import pynsitu as pyn

________
# CHOOSE PARAMETERS HERE

In [3]:
# TO CHOOSE 
drifters_sources = 'all_med_variational_10min_v0.nc'
spectral_decomp = True
low_pass = False
dt = '12h' #'nearestswath'
#dt = '10d' #'nearestswath'

_______
# Data 

In [4]:
# SWOT 250m browsing dataframe
dfs = browse_swot_250().reset_index()
# Drifter 
if spectral_decomp : 
    dr = xr.open_dataset(os.path.join(zarr_dir,'before_coloc','spectral_decomp_'+drifters_sources))
    drifters_sources = 'spectral_decomp_'+drifters_sources

if low_pass : 
    dr = xr.open_dataset(os.path.join(zarr_dir,'before_coloc','low_pass_'+drifters_sources))
    drifters_sources = 'low_pass_'+drifters_sources
    
else : 
    dr = xr.open_dataset(os.path.join(drifters_dir,'L2', drifters_sources))

dr_key = dr[['cruise_id', 'drifter_type']]
dr = dr.where(dr.gap_mask==1)

_____
# Select drifters under SWOT swath

In [5]:
def sel_drifters(dr, dt, swath, cycle):
    dfs_ = dfs.where((dfs.pass_number==swath)&(dfs.cycle_number==cycle)).dropna()
    dss = xr.open_dataset(dfs.where((dfs.pass_number==swath)&(dfs.cycle_number==cycle)).dropna().file.values[0])
    
    #time
    if dt == 'nearestswath':
        dr_ = dr.sel(datetime = slice(pd.to_datetime(dfs_.start_time_cut).values[0], pd.to_datetime(dfs_.end_time_cut).values[0]))
    else : 
        dr_ = dr.sel(datetime = slice((pd.to_datetime(dfs_.time)-pd.Timedelta(dt)).values[0], (pd.to_datetime(dfs_.time)+pd.Timedelta(dt)).values[0]))

    dr_= add_mask_inside_swot(dss, dr_)

    #under swath
    dr_ = dr_.where(dr_["inside_left"]+dr_["inside_right"])
    dfr_ = dr_.to_dataframe().reset_index().dropna()
    #time to nearest swot
    dfr_['time_to_swot']=(dfr_.datetime-dfs_.time.values[0]).abs()

    # stats for each drifters
    dfrs_=pd.DataFrame()
    dfrs_['time_to_swot_min'] = dfr_.groupby('drifter_id').time_to_swot.min()
    dfrs_['time_to_swot_max'] = dfr_.groupby('drifter_id').time_to_swot.max()
    dfrs_['point_number'] = dfr_.groupby('drifter_id').datetime.count()
    dfrs_['cycle_number'] = int(dfs_.cycle_number.values[0])
    dfrs_['cycle_date'] = dfs_.time.values[0]
    dfrs_['pass_number'] = int(dfs_.pass_number.values[0])
    return dfs_, dfr_, dfrs_.reset_index()

In [6]:
# EXAMPLE
dfs_, dfr_, dfrs_ = sel_drifters(dr, '12h', 3, 500)
dfr_

/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)


,drifter_id,datetime,x,y,cruise_id,lonc,latc,longitude,latitude,velocity_east,...,diurnal_acceleration_east,diurnal_acceleration_north,semidiurnal_velocity_east,semidiurnal_velocity_north,semidiurnal_acceleration_east,semidiurnal_acceleration_north,inside_left,inside_right,drifter_type,time_to_swot
720,300534060113380,2023-04-23 08:20:00,-95763.466447,213503.097325,C-SWOT,6.1710,38.3692,5.045020,40.286939,0.110045,...,-7.948046e-08,-3.099236e-08,0.002360,-0.002130,1.961277e-07,-4.068567e-07,0.0,1.0,SVP,0 days 11:56:27.404650304
721,300534060113380,2023-04-23 08:30:00,-95697.436335,213457.249599,C-SWOT,6.1710,38.3692,5.045803,40.286533,0.109970,...,-8.241029e-08,-2.740779e-08,0.002468,-0.002365,1.629490e-07,-3.753715e-07,0.0,1.0,SVP,0 days 11:46:27.404650304
722,300534060113380,2023-04-23 08:40:00,-95631.502627,213410.066236,C-SWOT,6.1710,38.3692,5.046585,40.286116,0.109726,...,-8.516949e-08,-2.385145e-08,0.002556,-0.002581,1.284604e-07,-3.413451e-07,0.0,1.0,SVP,0 days 11:36:27.404650304
723,300534060113380,2023-04-23 08:50:00,-95565.765514,213361.468400,C-SWOT,6.1710,38.3692,5.047365,40.285685,0.109319,...,-8.775634e-08,-2.032975e-08,0.002622,-0.002775,9.294403e-08,-3.050561e-07,0.0,1.0,SVP,0 days 11:26:27.404650304
724,300534060113380,2023-04-23 09:00:00,-95500.319381,213311.384567,C-SWOT,6.1710,38.3692,5.048142,40.285242,0.108763,...,-9.016949e-08,-1.684891e-08,0.002667,-0.002947,5.669014e-08,-2.667981e-07,0.0,1.0,SVP,0 days 11:16:27.404650304
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23134,300534060015760,2023-04-24 00:00:00,24639.476541,29991.547202,C-SWOT,4.1131,40.0546,4.403015,40.324339,0.109176,...,-4.894251e-07,3.786480e-09,0.009557,0.003591,4.618769e-07,-1.453579e-06,1.0,0.0,SVP,0 days 03:43:32.595349696
23135,300534060015760,2023-04-24 00:10:00,24705.129123,29923.666597,C-SWOT,4.1131,40.0546,4.403785,40.323726,0.110089,...,-4.803860e-07,-1.927687e-08,0.009797,0.002707,3.372459e-07,-1.487438e-06,1.0,0.0,SVP,0 days 03:53:32.595349696
23136,300534060015760,2023-04-24 00:20:00,24771.583811,29857.989477,C-SWOT,4.1131,40.0546,4.404564,40.323132,0.111814,...,-4.704916e-07,-4.236464e-08,0.009961,0.001806,2.104935e-07,-1.509900e-06,1.0,0.0,SVP,0 days 04:03:32.595349696
23137,300534060015760,2023-04-24 00:30:00,24839.306483,29794.571215,C-SWOT,4.1131,40.0546,4.405359,40.322559,0.114231,...,-4.597550e-07,-6.543751e-08,0.010049,0.000895,8.259578e-08,-1.520838e-06,1.0,0.0,SVP,0 days 04:13:32.595349696


_____
# Create and store dataset

In [6]:
D = []
for swath in [3,16]:
    for cycle in dfs.where(dfs.pass_number==swath).dropna().cycle_number:
        dfs_, dfr_, dfrs_ = sel_drifters(dr, dt, swath, cycle)
        dfr_['cycle_number'] = int(dfs_.cycle_number.values[0])
        dfr_['cycle_date'] = dfs_.time.values[0]
        dfr_['pass_number'] = int(dfs_.pass_number.values[0])
        D.append(dfr_)
        print(cycle)
        
df = pd.concat(D).set_index('pass_number').reset_index()
df['row_number'] = np.arange(len(df))
df = df.set_index('row_number')

# Store
df.to_csv(os.path.join(zarr_dir,'drifters', f'drifterscoloc_{dt}_'+drifters_sources.replace('.nc', '.csv')))

478.0
479.0
480.0
481.0
482.0
483.0
484.0
485.0
486.0
487.0
488.0
489.0
490.0
491.0
492.0
493.0
494.0
495.0
496.0
497.0
499.0
500.0
501.0
502.0
503.0
504.0
505.0
506.0
507.0
508.0
509.0
510.0
511.0
512.0
513.0
514.0
515.0
516.0
517.0
518.0
519.0
520.0
521.0
522.0
523.0
524.0
525.0
529.0
530.0
531.0
532.0
533.0
534.0
535.0
536.0
537.0
538.0
539.0
540.0
541.0
542.0
543.0
544.0
545.0
546.0
547.0
548.0
549.0
550.0
551.0
552.0
553.0
554.0
555.0
556.0
557.0
558.0
559.0
560.0
561.0
562.0
563.0
564.0
565.0
567.0
569.0
570.0
571.0
573.0
574.0
575.0
576.0
577.0
578.0
478.0
479.0
481.0
482.0
483.0
484.0
485.0
486.0
487.0
488.0
489.0
490.0
491.0
492.0
493.0
494.0
495.0
496.0
497.0
498.0
499.0
500.0
501.0
502.0
503.0
504.0
505.0
506.0
507.0
509.0
510.0
511.0
512.0
514.0
515.0
516.0
517.0
518.0
519.0
520.0
521.0
522.0
523.0
524.0
525.0
529.0
530.0
531.0
532.0
533.0
535.0
536.0
537.0
538.0
539.0
540.0
541.0
542.0
543.0
544.0
545.0
546.0
547.0
548.0
549.0
550.0
551.0
552.0
553.0
554.0
555.0
556.0
557.

In [8]:
D = []
for swath in [3,16]:
    for cycle in dfs.where(dfs.pass_number==swath).dropna().cycle_number:
        dfs_, dfr_, dfrs_ = sel_drifters(dr, dt, swath, cycle)
        D.append(dfrs_)
df_stats = pd.concat(D)
df_stats.to_csv(os.path.join(zarr_dir,'drifters', f'driftersstats_{dt}_'+drifters_sources.replace('.nc', '.csv')))

/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails wh

In [7]:
df

,pass_number,drifter_id,datetime,x,y,cruise_id,lonc,latc,longitude,latitude,...,semidiurnal_velocity_east,semidiurnal_velocity_north,semidiurnal_acceleration_east,semidiurnal_acceleration_north,inside_left,inside_right,drifter_type,time_to_swot,cycle_number,cycle_date
row_number,,,,,,,,,,,,,,,,,,,,,
0,3,300534060315840,2023-04-01 11:50:00,-98662.911488,19783.062142,C-SWOT,6.0070,42.0490,4.811970,42.220893,...,0.009955,-0.009880,-2.326338e-07,-0.000002,1.0,0.0,SVP,0 days 11:52:32.696536,478,2023-04-01 23:42:32.696536000
1,3,300534060315840,2023-04-01 12:00:00,-98623.960276,19842.195526,C-SWOT,6.0070,42.0490,4.812432,42.221430,...,0.009778,-0.011003,-3.583966e-07,-0.000002,1.0,0.0,SVP,0 days 11:42:32.696536,478,2023-04-01 23:42:32.696536000
2,3,300534060315840,2023-04-01 12:10:00,-98583.178329,19897.134982,C-SWOT,6.0070,42.0490,4.812916,42.221930,...,0.009525,-0.012047,-4.817391e-07,-0.000002,1.0,0.0,SVP,0 days 11:32:32.696536,478,2023-04-01 23:42:32.696536000
3,3,300534060315840,2023-04-01 12:20:00,-98541.006388,19948.108049,C-SWOT,6.0070,42.0490,4.813418,42.222394,...,0.009200,-0.013003,-6.017149e-07,-0.000002,1.0,0.0,SVP,0 days 11:22:32.696536,478,2023-04-01 23:42:32.696536000
4,3,300534060315840,2023-04-01 12:30:00,-98497.853233,19995.403935,C-SWOT,6.0070,42.0490,4.813933,42.222825,...,0.008803,-0.013863,-7.174011e-07,-0.000001,1.0,0.0,SVP,0 days 11:12:32.696536,478,2023-04-01 23:42:32.696536000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376718,16,300534060015760,2023-07-10 06:10:00,-145207.110838,-28177.646918,C-SWOT,4.1131,40.0546,2.417786,39.788394,...,-0.040939,-0.009495,-1.738689e-06,0.000005,1.0,0.0,SVP,0 days 11:13:38.131544384,577,2023-07-09 18:56:21.868455616
376719,16,300534060015760,2023-07-10 06:20:00,-145283.518098,-28121.682437,C-SWOT,4.1131,40.0546,2.416881,39.788885,...,-0.041829,-0.006689,-1.221971e-06,0.000005,1.0,0.0,SVP,0 days 11:23:38.131544384,577,2023-07-09 18:56:21.868455616
376720,16,300534060015760,2023-07-10 06:30:00,-145362.329231,-28063.105917,C-SWOT,4.1131,40.0546,2.415948,39.789399,...,-0.042406,-0.003842,-6.963034e-07,0.000005,1.0,0.0,SVP,0 days 11:33:38.131544384,577,2023-07-09 18:56:21.868455616
